In [11]:
import pandas as pd
import pandas_ta as ta

## Loading the Data

In [54]:
df = pd.read_csv('btc_1hour_cleaned.csv')

In [55]:
df.head(2)

,Open time,Open,High,Low,Close,Volume,Close time,Quote asset volume,Number of trades,Taker buy base asset volume,Taker buy quote asset volume,Ignore
0,2018-01-01 00:00:00,13715.65,13715.65,13400.01,13529.01,443.356199,2018-01-01 00:59:59.999,5.993910e+06,5228,228.521921,3.090541e+06,0
1,2018-01-01 01:00:00,13528.99,13595.89,13155.38,13203.06,383.697006,2018-01-01 01:59:59.999,5.154522e+06,4534,180.840403,2.430449e+06,0


## Setting the correct formats and cleaning Dataframe

In [56]:
df['Open time'] = pd.to_datetime(df['Open time'])
df['Close time'] = pd.to_datetime(df['Close time'])

In [57]:
df = df.set_index('Open time')

In [58]:
df.head(3)

,Open,High,Low,Close,Volume,Close time,Quote asset volume,Number of trades,Taker buy base asset volume,Taker buy quote asset volume,Ignore
Open time,,,,,,,,,,,
2018-01-01 00:00:00,13715.65,13715.65,13400.01,13529.01,443.356199,2018-01-01 00:59:59.999,5.993910e+06,5228,228.521921,3.090541e+06,0
2018-01-01 01:00:00,13528.99,13595.89,13155.38,13203.06,383.697006,2018-01-01 01:59:59.999,5.154522e+06,4534,180.840403,2.430449e+06,0
2018-01-01 02:00:00,13203.00,13418.43,13200.00,13330.18,429.064572,2018-01-01 02:59:59.999,5.710192e+06,4887,192.237935,2.558505e+06,0


In [59]:
df.shape

(71588, 11)

# Feature Engineering

## Functions for features: Strategy 1 + regime

In [60]:
# ROC
def compute_roc(df):
    df['roc_10'] = df['Close'].pct_change(periods=10)
    df['roc_21'] = df['Close'].pct_change(periods=21)
    return df

# MACD Histogram
def compute_macd(df):
    macd = ta.macd(df['Close'], fast=12, slow=26, signal=9)
    df['macd_histogram'] = macd['MACDh_12_26_9']
    return df

# ADX - Used as a feature and for regime detection
def compute_adx(df):
    df['adx'] = ta.adx(df['High'], df['Low'], df['Close'], length=14)['ADX_14']
    return df

# SMA 50 and SMA 200 is purely used for regime detection but not in feature matrix
def compute_regime(df):
    df['sma_50']  = df['Close'].rolling(50).mean()
    df['sma_200'] = df['Close'].rolling(200).mean()
    df['regime']  = (df['sma_50'] > df['sma_200']).astype(int).replace(0, -1)
    return df

# ATR 14
def compute_atr(df):
    df['atr_14'] = ta.atr(df['High'], df['Low'], df['Close'], length=14)
    return df

# NATR
def compute_natr(df):
    df['natr'] = ta.natr(df['High'], df['Low'], df['Close'], length=14)
    return df

## Pipeline

In [61]:
def feature_pipeline(df):
    steps = [
        # model features (go to X)
        compute_roc,
        compute_macd,
        compute_adx,
        compute_atr,
        compute_natr,
        # execution layer only (do not go to X)
        compute_regime
    ]
    for step in steps:
        df = step(df)
    return df

In [62]:
df = feature_pipeline(df)

In [63]:
df.head(20)

,Open,High,Low,Close,Volume,Close time,Quote asset volume,Number of trades,Taker buy base asset volume,Taker buy quote asset volume,Ignore,roc_10,roc_21,macd_histogram,adx,atr_14,natr,sma_50,sma_200,regime
Open time,,,,,,,,,,,,,,,,,,,,
2018-01-01 00:00:00,13715.65,13715.65,13400.01,13529.01,443.356199,2018-01-01 00:59:59.999,5.993910e+06,5228,228.521921,3.090541e+06,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1
2018-01-01 01:00:00,13528.99,13595.89,13155.38,13203.06,383.697006,2018-01-01 01:59:59.999,5.154522e+06,4534,180.840403,2.430449e+06,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1
2018-01-01 02:00:00,13203.00,13418.43,13200.00,13330.18,429.064572,2018-01-01 02:59:59.999,5.710192e+06,4887,192.237935,2.558505e+06,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1
2018-01-01 03:00:00,13330.26,13611.27,13290.00,13410.03,420.087030,2018-01-01 03:59:59.999,5.657448e+06,4789,137.918407,1.858041e+06,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1
2018-01-01 04:00:00,13434.98,13623.29,13322.15,13601.01,340.807329,2018-01-01 04:59:59.999,4.588047e+06,4563,172.957635,2.328058e+06,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1
2018-01-01 05:00:00,13615.20,13699.00,13526.50,13558.99,404.229046,2018-01-01 05:59:59.999,5.499055e+06,5086,142.331058,1.935710e+06,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1
2018-01-01 06:00:00,13539.00,13800.00,13510.00,13780.41,264.989684,2018-01-01 06:59:59.999,3.613408e+06,4072,126.077500,1.718753e+06,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1
2018-01-01 07:00:00,13780.00,13818.55,13555.02,13570.35,292.188777,2018-01-01 07:59:59.999,4.002026e+06,4340,147.150029,2.016275e+06,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1
2018-01-01 08:00:00,13569.98,13735.24,13400.00,13499.99,271.813553,2018-01-01 08:59:59.999,3.681944e+06,3733,122.809696,1.664737e+06,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1


# Creating y

In [64]:
# Lookahead window - change this value to test different horizons
N = 5

# Step 1 - Forward return
df['forward_return'] = df['Close'].pct_change(periods=N).shift(-N)

In [65]:
# Step 2 - Binary Label
df['label'] = (df['forward_return'] > 0).astype(int)
# 1 - BUY, 0 - SELL

In [66]:
df.head(2)

,Open,High,Low,Close,Volume,Close time,Quote asset volume,Number of trades,Taker buy base asset volume,Taker buy quote asset volume,...,roc_21,macd_histogram,adx,atr_14,natr,sma_50,sma_200,regime,forward_return,label
Open time,,,,,,,,,,,,,,,,,,,,,
2018-01-01 00:00:00,13715.65,13715.65,13400.01,13529.01,443.356199,2018-01-01 00:59:59.999,5.993910e+06,5228,228.521921,3.090541e+06,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1,0.002216,1
2018-01-01 01:00:00,13528.99,13595.89,13155.38,13203.06,383.697006,2018-01-01 01:59:59.999,5.154522e+06,4534,180.840403,2.430449e+06,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1,0.043728,1


In [67]:
df.shape

(71588, 22)

In [68]:
df.columns

Index(['Open', 'High', 'Low', 'Close', 'Volume', 'Close time',
       'Quote asset volume', 'Number of trades', 'Taker buy base asset volume',
       'Taker buy quote asset volume', 'Ignore', 'roc_10', 'roc_21',
       'macd_histogram', 'adx', 'atr_14', 'natr', 'sma_50', 'sma_200',
       'regime', 'forward_return', 'label'],
      dtype='str')

## Creating the lags

In [69]:
# Number of lags to create
N_lags = 5

feature_cols = ['Volume','roc_10', 'roc_21', 'macd_histogram', 'adx']

# Create lags for all feature columns
for col in feature_cols:
    for lag in range(1, N_lags + 1):
        df[f'{col}_lag{lag}'] = df[col].shift(lag)

# Drop NaN rows created by the lags
df = df.dropna(subset=feature_cols + [f'{col}_lag{i}' for col in feature_cols for i in range(1, N_lags + 1)] + ['label'])

In [70]:
df.shape

(71550, 47)

In [71]:
df.columns

Index(['Open', 'High', 'Low', 'Close', 'Volume', 'Close time',
       'Quote asset volume', 'Number of trades', 'Taker buy base asset volume',
       'Taker buy quote asset volume', 'Ignore', 'roc_10', 'roc_21',
       'macd_histogram', 'adx', 'atr_14', 'natr', 'sma_50', 'sma_200',
       'regime', 'forward_return', 'label', 'Volume_lag1', 'Volume_lag2',
       'Volume_lag3', 'Volume_lag4', 'Volume_lag5', 'roc_10_lag1',
       'roc_10_lag2', 'roc_10_lag3', 'roc_10_lag4', 'roc_10_lag5',
       'roc_21_lag1', 'roc_21_lag2', 'roc_21_lag3', 'roc_21_lag4',
       'roc_21_lag5', 'macd_histogram_lag1', 'macd_histogram_lag2',
       'macd_histogram_lag3', 'macd_histogram_lag4', 'macd_histogram_lag5',
       'adx_lag1', 'adx_lag2', 'adx_lag3', 'adx_lag4', 'adx_lag5'],
      dtype='str')

## Creating the Feature Matrix and y

In [73]:
X = df[['Volume','roc_10', 'roc_21', 'macd_histogram', 'adx',
                     'Volume_lag1', 'Volume_lag2', 'Volume_lag3', 'Volume_lag4', 'Volume_lag5',
                     'roc_10_lag1', 'roc_10_lag2', 'roc_10_lag3', 'roc_10_lag4', 'roc_10_lag5',
                     'roc_21_lag1', 'roc_21_lag2', 'roc_21_lag3', 'roc_21_lag4', 'roc_21_lag5',
                     'macd_histogram_lag1', 'macd_histogram_lag2', 'macd_histogram_lag3', 'macd_histogram_lag4', 'macd_histogram_lag5',
                     'adx_lag1', 'adx_lag2', 'adx_lag3', 'adx_lag4', 'adx_lag5']]

In [75]:
X.shape

(71550, 30)

In [76]:
y = df[['label']]

In [78]:
y.shape

(71550, 1)

# Training the Model

In [87]:
import lightgbm as lgb
from sklearn.model_selection import TimeSeriesSplit
import numpy as np
import time

# ================================================================
# STEP 1 — Define model parameters
# ================================================================

params = {
    'objective':        'binary',
    'metric':           'binary_logloss',
    'boosting_type':    'gbdt',
    'learning_rate':    0.05,
    'num_leaves':       31,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq':     5,
    'verbose':          -1
}

# ================================================================
# STEP 2 — Walk-Forward Cross Validation
# ================================================================

tscv = TimeSeriesSplit(n_splits=5)

all_predictions = []

for fold, (train_idx, test_idx) in enumerate(tscv.split(X)):

    start = time.time()

    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    lgb_train = lgb.Dataset(X_train, label=y_train)
    lgb_test  = lgb.Dataset(X_test,  label=y_test, reference=lgb_train)

    model = lgb.train(
        params          = params,
        train_set       = lgb_train,
        num_boost_round = 300,
        valid_sets      = [lgb_test],
        callbacks       = [lgb.early_stopping(50), lgb.log_evaluation(50)]
    )

    fold_preds = model.predict(X_test)
    fold_df = y_test.copy()
    fold_df = fold_df.assign(pred_proba=fold_preds)
    fold_df['fold'] = fold + 1
    all_predictions.append(fold_df)

    elapsed = time.time() - start
    print(f'Fold {fold+1} done — {len(test_idx)} bars — {elapsed:.1f}s')

# ================================================================
# STEP 3 — Collect all predictions
# ================================================================

results = pd.concat(all_predictions)
print(results.head())
print(results.shape)

Training until validation scores don't improve for 50 rounds
[50]	valid_0's binary_logloss: 0.691207
Early stopping, best iteration is:
[18]	valid_0's binary_logloss: 0.689923
Fold 1 done — 11925 bars — 0.4s
Training until validation scores don't improve for 50 rounds
[50]	valid_0's binary_logloss: 0.693122
Early stopping, best iteration is:
[13]	valid_0's binary_logloss: 0.690732
Fold 2 done — 11925 bars — 0.3s
Training until validation scores don't improve for 50 rounds
[50]	valid_0's binary_logloss: 0.694732
Early stopping, best iteration is:
[7]	valid_0's binary_logloss: 0.693366
Fold 3 done — 11925 bars — 0.3s
Training until validation scores don't improve for 50 rounds
[50]	valid_0's binary_logloss: 0.690815
Early stopping, best iteration is:
[21]	valid_0's binary_logloss: 0.690116
Fold 4 done — 11925 bars — 0.5s
Training until validation scores don't improve for 50 rounds
[50]	valid_0's binary_logloss: 0.693026
Early stopping, best iteration is:
[15]	valid_0's binary_logloss: 0.

In [88]:
results

,label,pred_proba,fold
Open time,,,
2019-05-17 17:00:00,1,0.491558,1
2019-05-17 18:00:00,1,0.493925,1
2019-05-17 19:00:00,1,0.488483,1
2019-05-17 20:00:00,1,0.515783,1
2019-05-17 21:00:00,1,0.526965,1
...,...,...,...
2026-03-07 17:00:00,0,0.513068,5
2026-03-07 18:00:00,0,0.553708,5
2026-03-07 19:00:00,0,0.552612,5
